Analyze sensors difference between monitoring stations and low-cost sensors.

Time period is 10222023~11222023.

Data readings are averaged every 60 minutes.

The whole map is separated into a 15x15 grid.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Data Preprocessing

## Purple Air

In [2]:
def clean_timestamp(df):
    # df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_convert("US/Pacific")
    ts = pd.to_datetime(df["timestamp"], format="mixed")
    df["year"] = ts.dt.year
    df["month"] = ts.dt.month
    df["day"] = ts.dt.day
    df["hour"] = ts.dt.hour
    df["minute"] = ts.dt.minute

    df = df.drop(columns=["timestamp"])
    return df

def average_60min(df, col_name="pm25"):
    df = df[["year", "month", "day", "hour", col_name]]
    df = df.groupby(["year", "month", "day", "hour"]).mean().reset_index()
    return df

In [3]:
dir_path = "../SJVAir_Data/raw/10222023_11222023/purpleair/"
for file_name in os.listdir(dir_path):
    if not file_name.endswith(".csv"):
        continue
    df = pd.read_csv(dir_path + file_name)

    # discard if ab channels' difference is greater than 100
    diff = np.mean(df["pm25"][0::2].values) - np.mean(df["pm25"][1::2].values)
    if np.abs(diff) > 100:
        print("Discard because ab channel difference")
        continue
    
    df = average_60min(clean_timestamp(df))

    # discard if there are missing values
    if len(df) < 744:
        print(len(df))
        continue
    else:
        df.to_csv("./data/processed/purpleair/{}".format(file_name), index=False)

Discard because ab channel difference
622
676
Discard because ab channel difference
151
218


In [4]:
# add location
fresno_sensor = pd.read_csv("../SJVAir_Data/fresno_sensors.csv")

dir_path = "./data/processed/purpleair/"
for file_name in os.listdir(dir_path):
    if not file_name.endswith(".csv"):
        continue

    sensor_id = file_name.split(".")[0]
    df = pd.read_csv(dir_path + file_name)
    sensor_loc = fresno_sensor[fresno_sensor["id"] == sensor_id][["longitude", "latitude"]].to_numpy()[0]
    df["longitude"] = sensor_loc[0]
    df["latitude"] = sensor_loc[1]
    df.to_csv(dir_path + file_name, index=False)

## AQview and AirNow

In [34]:
date_range = pd.date_range(start="2023-10-22-00-00-00", end="2023-11-21-23-00-00", freq="60min")
date_range = pd.DataFrame(date_range, columns=["timestamp"])
date_range = clean_timestamp(date_range)
date_range

,year,month,day,hour,minute
0,2023,10,22,0,0
1,2023,10,22,1,0
2,2023,10,22,2,0
3,2023,10,22,3,0
4,2023,10,22,4,0
...,...,...,...,...,...
739,2023,11,21,19,0
740,2023,11,21,20,0
741,2023,11,21,21,0
742,2023,11,21,22,0


In [35]:
dir_path = "../SJVAir_Data/raw/10222023_11222023/aqan/"
for file_name in os.listdir(dir_path):
    if not file_name.endswith(".csv"):
        continue

    df = pd.read_csv(dir_path + file_name)

    # discard if ab channels' difference is greater than 100
    diff = np.mean(df["pm25"][0::2].values) - np.mean(df["pm25"][1::2].values)
    if diff > 100:
        print("Discard because ab channel difference")
        continue

    df = average_60min(clean_timestamp(df))

    # discard if there are more than 10% missing values
    if len(df) < len(date_range) * 0.9:
        print(len(df))
        continue
    df = pd.merge(date_range, df, on=["year", "month", "day", "hour"], how="left")
    df["pm25"] = df["pm25"].interpolate(method="linear")
    df.to_csv("./data/processed/aqan/{}".format(file_name), index=False)

451
480
648


In [36]:
# add location
fresno_sensor = pd.read_csv("../SJVAir_Data/fresno_sensors.csv")

dir_path = "./data/processed/aqan/"
for file_name in os.listdir(dir_path):
    if not file_name.endswith(".csv"):
        continue

    sensor_id = file_name.split(".")[0]
    df = pd.read_csv(dir_path + file_name)
    sensor_loc = fresno_sensor[fresno_sensor["id"] == sensor_id][["longitude", "latitude"]].to_numpy()[0]
    df["longitude"] = sensor_loc[0]
    df["latitude"] = sensor_loc[1]
    df.to_csv(dir_path + file_name, index=False)